# 1. Libraries load

In [ ]:
import pandas as pd
import sqlite3

import pandas as pd
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
from IPython.display import Markdown as md
from matplotlib.patches import Patch
import matplotlib.ticker as ticker

# 2. Connection and data load

In [ ]:
conn = sqlite3.connect("data/weather.db")
df = pd.read_sql("SELECT * FROM weather_air_quality", conn)
conn.close()

# drop potential duplicates
df = df.drop_duplicates(subset=["city", "downloaded_utc"])

# 3. Analysis 

In [ ]:
# Basic info
print(f"Total measurements: {len(df)}")
print(f"Cities: {df["city"].nunique()}")
print(f"Time range: {df["downloaded_utc"].min()} to {df["downloaded_utc"].max()}")
df.head()

In [ ]:
# Missing/invalid data
print("Missing PM2.5 data:")
print(df["air_quality_category"].value_counts())

In [ ]:
# cities with no PM2.5 measurements
missing_pm25 = df.groupby("city").agg(pm25_measurements=("pm25_last_value[ug/m3]", "count"))
missing_pm25 = len(missing_pm25[missing_pm25["pm25_measurements"] == 0])

print(f"Insight: {missing_pm25} out of {len(df)} cities have incomplete PM2.5 coverage due to sensor availability")

In [ ]:
# Latest measurement per city
latest = df.sort_values("downloaded_utc").groupby("city").last().reset_index()
latest = latest[latest["pm25_last_value[ug/m3]"].notna()]
latest = latest.sort_values(by="pm25_last_value[ug/m3]", ascending=False)

sns.set_theme(style="white")
f, ax1 = plt.subplots(1, 1, figsize=(20, 8))

category_order = ["Good", "Moderate", "Unhealthy for sensitive groups", "Unhealthy", "Unknown"]
category_colors = {
    "Good": "green", 
    "Moderate": "orange", 
    "Unhealthy for sensitive groups": "blue", 
    "Unhealthy": "red", 
    "Unknown": "gray"
}

# AX1: PM2.5 & categories
(
    so.Plot(latest, x="city", y="pm25_last_value[ug/m3]", color="air_quality_category")
    .add(
        so.Bar(alpha=0.8)
        )
    .scale(
        color=category_colors,
        y=so.Continuous().tick(every=5),
        x=so.Nominal(),
        )
    .label(title="Current PM2.5 Levels by City", y="PM2.5 (μg/m³)", x="City")
    .on(ax1)
    .plot()
)

# Manual legend (in correct order)
legend_handles = [
    Patch(facecolor=category_colors["Good"], label="Good"),
    Patch(facecolor=category_colors["Moderate"], label="Moderate"),
    Patch(facecolor=category_colors["Unhealthy for sensitive groups"], label="Unhealthy for sensitive groups"),
    Patch(facecolor=category_colors["Unhealthy"], label="Unhealthy"),
    plt.Line2D([0], [0], color="green", linestyle="-", alpha=0.5, label="US EPA Good upper limit"),
    plt.Line2D([0], [0], color="orange", linestyle="-", alpha=0.5, label="US EPA Moderate upper limit"),
    plt.Line2D([0], [0], color="blue", linestyle="-", alpha=0.5, label="US EPA Unhealthy for sensitive groups upper limit")
]

ax1.legend(handles=legend_handles, loc="upper right")

# Remove auto-generated legend from seaborn.objects
f.legends.clear()

ax1.xaxis.set_tick_params(which="both", left=True, length=4, rotation=90)
ax1.yaxis.set_tick_params(which="both", left=True, length=4)
ax1.yaxis.grid(True, linestyle="--", alpha=0.6)
ax1.xaxis.grid(True, linestyle="--", alpha=0.6)
ax1.axhline(y=12, color="green", linestyle="-", alpha=0.5, label="US EPA Good upper limit")
ax1.axhline(y=35.4, color="orange", linestyle="-", alpha=0.5, label="US EPA Moderate upper limit")
ax1.axhline(y=55.4, color="blue", linestyle="-", alpha=0.5, label="US EPA Unhealthy for sensitive groups upper limit")
ax1.set_axisbelow(True)
ax1.yaxis.set_major_formatter(ticker.ScalarFormatter())

plt.tight_layout()
plt.show()

In [ ]:
worst_city = latest.sort_values(by="pm25_last_value[ug/m3]", ascending=False).head(1)[["city", "pm25_last_value[ug/m3]", "air_quality_category"]]

print(f"Insight: {worst_city["city"].item()} has the worst air quality with PM2.5 at {worst_city["pm25_last_value[ug/m3]"].item()} μg/m³, which is '{worst_city["air_quality_category"].item()}' category")

In [ ]:
# wind and PM2.5 relation
latest = df.sort_values("downloaded_utc").groupby("city").last().reset_index()
latest = latest[latest["pm25_last_value[ug/m3]"].notna()]
latest = latest.sort_values(by="pm25_last_value[ug/m3]", ascending=False)

sns.set_theme(style="white")
f, ax1 = plt.subplots(1, 1, figsize=(8, 8))

category_order = ["Good", "Moderate", "Unhealthy for sensitive groups", "Unhealthy", "Unknown"]
category_colors = {
    "Good": "green", 
    "Moderate": "orange", 
    "Unhealthy for sensitive groups": "blue", 
    "Unhealthy": "red", 
    "Unknown": "gray"
}

# AX1: wind speed & PM2.5
(
    so.Plot(latest, x="wind_speed", y="pm25_last_value[ug/m3]")
    .add(
        so.Dot(alpha=0.8),
        color="air_quality_category",
        pointsize="wind_speed",
        )
    .add(
            so.Line(color="#444", linewidth=1.5, alpha=0.7), # Stały kolor
            so.PolyFit(order=1),
            x="wind_speed", y="pm25_last_value[ug/m3]" # Jawne powtórzenie x i y
        )
    .scale(
        color=so.Nominal(category_colors),
        y=so.Continuous().tick(every=5),
        x=so.Continuous().tick(every=0.5),
        )
    .label(title="Current PM2.5 Levels by City", y="PM2.5 (μg/m³)", x="Wind speed [m/s]")
    .on(ax1)
    .plot()
)

# Remove auto-generated legend from seaborn.objects
f.legends.clear()

ax1.xaxis.set_tick_params(which="both", left=True, length=4, rotation=90)
ax1.yaxis.set_tick_params(which="both", left=True, length=4)
ax1.yaxis.grid(True, linestyle="--", alpha=0.6)
ax1.xaxis.grid(True, linestyle="--", alpha=0.6)

ax1.set_axisbelow(True)
ax1.yaxis.set_major_formatter(ticker.ScalarFormatter())

plt.tight_layout()
plt.show()

In [ ]:
corr = latest[["wind_speed", "pm25_last_value[ug/m3]", ]].corr(method="pearson").iloc[-1,0].round(2)

print(f"Insight: Wind speed shows weak correlation with PM2.5 (r={corr}), suggesting other factors (traffic, industry) play larger role")

In [ ]:
category_counts = df.sort_values("downloaded_utc").groupby("city").last().reset_index()
category_counts = category_counts["air_quality_category"].value_counts(sort=True, ascending=False).reset_index()

category_order = ["Good", "Moderate", "Unhealthy for sensitive groups", "Unhealthy", "Unknown"]
category_colors = {
    "Good": "green", 
    "Moderate": "orange", 
    "Unhealthy for sensitive groups": "blue", 
    "Unhealthy": "red", 
    "Unknown": "gray"
}


sns.set_theme(style="white")
f, ax1 = plt.subplots(1, 1, figsize=(8, 8))

# AX1: categories counts
(
    so.Plot(category_counts, x="air_quality_category", y="count", color="air_quality_category")
    .add(
        so.Bar(alpha=0.8)
        )
    .scale(
        y=so.Continuous().tick(),
        x=so.Nominal(),
        color=so.Nominal(category_colors)
        )
    .label(title="Air Quality Category Distribution Across Cities", y="Number of Cities", x="Category")
    .on(ax1)
    .plot()
)

# Manual legend (in correct order)
legend_handles = [
    Patch(facecolor=category_colors["Good"], label="Good"),
    Patch(facecolor=category_colors["Moderate"], label="Moderate"),
    Patch(facecolor=category_colors["Unhealthy for sensitive groups"], label="Unhealthy for sensitive groups"),
    Patch(facecolor=category_colors["Unhealthy"], label="Unhealthy"),
    Patch(facecolor=category_colors["Unknown"], label="Unknown")
]

ax1.legend(handles=legend_handles, loc="upper right")

# Remove auto-generated legend from seaborn.objects
f.legends.clear()

ax1.xaxis.set_tick_params(which="both", left=True, length=4, rotation=90)
ax1.yaxis.set_tick_params(which="both", left=True, length=4)
ax1.yaxis.grid(True, linestyle="--", alpha=0.6)
ax1.xaxis.grid(True, linestyle="--", alpha=0.6)
ax1.set_axisbelow(True)
ax1.yaxis.set_major_formatter(ticker.ScalarFormatter())

plt.tight_layout()
plt.show()

In [ ]:
all_categories = pd.Series(["Good", "Moderate", "Unhealthy for sensitive groups", "Unhealthy", "Unknown"], name="air_quality_category")
category_counts = pd.merge(all_categories, category_counts, how="left").fillna(0)
category_counts["count"] = category_counts["count"].astype("Int32")

all = category_counts["count"].sum().item()
good = category_counts.loc[category_counts["air_quality_category"] == "Good"]["count"].item()
moderate = category_counts.loc[category_counts["air_quality_category"] == "Moderate"]["count"].item()
unhealthy_sensitive = category_counts.loc[category_counts["air_quality_category"] == "Unhealthy for sensitive groups"]["count"].item()
unhealthy = category_counts.loc[category_counts["air_quality_category"] == "Unhealthy"]["count"].item()
unknown = category_counts.loc[category_counts["air_quality_category"] == "Unknown"]["count"].item()
print(f"Insight:")
print(f"- {(good / all) * 100:.2f}% cities have Good air quality")
print(f"- {(moderate / all) * 100:.2f}% cities have Moderate air quality")
print(f"- {(unhealthy_sensitive / all) * 100:.2f}% cities have Unhealthy for sensitive groups air quality")
print(f"- {(unhealthy / all) * 100:.2f}% cities have Unhealthy air quality")
print(f"- {(unknown / all) * 100:.2f}% cities have Unknown air quality - due to the missing sensors, lack of data in OpenAQ API or missing measurements in the 24h before data was collected")


In [ ]:
# Summary table
summary = latest[['city', 'temp', 'humidity', 'pressure', 'wind_speed', 
                  'pm25_last_value[ug/m3]', 'air_quality_category']].sort_values('pm25_last_value[ug/m3]', ascending=False)

print("Current Weather & Air Quality Summary:")
print(summary.to_string(index=False))

# Descriptive stats
print("\nStatistics:")
print(latest[['temp', 'humidity', 'pm25_last_value[ug/m3]']].describe().T)

In [ ]:
avg_temp = df["temp"].mean().round(2)
city_temp = df[["temp", "city"]].sort_values(by="temp")

markdown_content = f"""
## Key Findings

1. **Air Quality:**

   - **{worst_city["city"].item()}** has the worst air quality (PM2.5: **{worst_city["pm25_last_value[ug/m3]"].item()} μg/m³**, '{worst_city["air_quality_category"].item()}' category)
   - **{good}** out of **{len(latest)}** cities with data meet US EPA "Good" standards (PM2.5 ≤ 12 μg/m³)

2. **Temperature:**
   - Average temperature across cities: **{avg_temp}°C**
   - Warmest: **{city_temp.iloc[-1,1]}** ({city_temp.iloc[-1,0]}°C), Coldest: **{city_temp.iloc[0,0]}** ({city_temp.iloc[0,1]}°C)

3. **Data Quality:**
   - PM2.5 data not available for **{missing_pm25}/{len(df)}** cities (**{(1 - (missing_pm25 / len(df)))*100:.2f}%** coverage)
   - The missing data primarily concern cities that are not regional capitals, which is likely due to sensor availability

4. **Correlations:**
   - Weak correlation between wind speed and PM2.5 (r=**{corr}**)

## Recommendations

- Cities with "Unhealthy" air quality should issue public health advisories
- Invest in additional PM2.5 sensors in cities with missing data
- Monitor PM2.5 trends during high-traffic hours (if time series available)
"""

md(markdown_content)